# Oracle Eval — Gold Responses từ `fnb_dataset_test.json`

**Mục đích:** Chạy pipeline eval (Faithfulness + Expansion + Marketing Vibe) trực tiếp trên
các `response` chuẩn trong test set, **không sinh bài mới**.

Kết quả này là **trần (ceiling)** để so sánh:
```
oracle (gold)   → trần, ví dụ 0.80
baseline GPT    → tham chiếu, ví dụ 0.52
trained SLM     → mục tiêu, hy vọng → 0.70+
```

**Output:** append vào `results/fnb_eval_summary.csv` với `model_role=oracle`.

In [9]:
# %pip install -q deepeval openai pandas python-dotenv

In [10]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

# ===== Hyperparameters =====
DATASET_REL = "dataset/fnb_dataset_test.json"
SUMMARY_REL = "results/fnb_eval_summary.csv"   # shared with baseline + trained runs
RUNS_REL    = "results/runs"

EVAL_PROGRESS_CHUNK = 10

def _env(k: str) -> str:
    return (os.getenv(k) or "").strip()

# ---- Judge — Azure only ----
JUDGE_MODEL            = (_env("JUDGE_MODEL") or _env("OPENAI_MODEL") or "gpt-4o-mini").strip()
JUDGE_AZURE_ENDPOINT   = (_env("JUDGE_AZURE_ENDPOINT") or _env("AZURE_OPENAI_ENDPOINT")).rstrip("/")
JUDGE_AZURE_API_KEY    = _env("JUDGE_AZURE_API_KEY") or _env("AZURE_OPENAI_API_KEY")
JUDGE_AZURE_API_VERSION = _env("JUDGE_AZURE_API_VERSION") or _env("OPENAI_API_VERSION") or "2024-08-01-preview"
if not (JUDGE_AZURE_ENDPOINT and JUDGE_AZURE_API_KEY):
    raise RuntimeError("Judge cần JUDGE_AZURE_ENDPOINT + JUDGE_AZURE_API_KEY (hoặc AZURE_OPENAI_*).")

os.environ["USE_AZURE_OPENAI"] = "true"
os.environ.setdefault("OPENAI_API_VERSION", JUDGE_AZURE_API_VERSION)
os.environ.pop("USE_OPENAI_MODEL", None)
os.environ["OPENAI_MODEL"] = "azure"

DATASET_PATH = ROOT / DATASET_REL
SUMMARY_PATH = ROOT / SUMMARY_REL
RUNS_DIR     = ROOT / RUNS_REL

print("Judge:", JUDGE_MODEL, "@", JUDGE_AZURE_ENDPOINT)
print("Dataset:", DATASET_PATH)
print("Summary:", SUMMARY_PATH)

Judge: gpt-5.4 @ https://vqnhan-poc.openai.azure.com
Dataset: D:\Github\mcs-train-content-model\dataset\fnb_dataset_test.json
Summary: D:\Github\mcs-train-content-model\results\fnb_eval_summary.csv


In [11]:
from openai import AzureOpenAI

_judge = AzureOpenAI(
    azure_endpoint=JUDGE_AZURE_ENDPOINT,
    api_key=JUDGE_AZURE_API_KEY,
    api_version=JUDGE_AZURE_API_VERSION,
)
_judge_model_id = JUDGE_MODEL

_r = _judge.chat.completions.create(
    model=_judge_model_id,
    messages=[{"role": "user", "content": "Xin chào"}],
)
print("Judge OK:", (_r.choices[0].message.content or "").strip()[:80])

Judge OK: Xin chào! Mình có thể giúp gì cho bạn hôm nay?


## Load gold responses — không sinh bài

In [12]:
import json
from typing import List, Dict

def load_oracle_cases(json_path: Path) -> List[Dict[str, str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Dataset phải là mảng JSON.")
    cases = []
    for i, row in enumerate(data):
        response = str(row.get("response", "")).strip()
        if not response:
            print(f"  ⚠ row {i}: thiếu 'response', bỏ qua.")
            continue
        cases.append({
            "case_id":      str(i),
            "input_title":  str(row.get("instruction", "")),
            "seed_content": str(row.get("input", "")),
            "actual_output": response,
        })
    return cases


cases = load_oracle_cases(DATASET_PATH)
print(f"Oracle cases: {len(cases)} | dataset: {DATASET_PATH.name}")
print("\nSample case_0 output (first 200 chars):")
print(cases[0]["actual_output"][:200])

Oracle cases: 100 | dataset: fnb_dataset_test.json

Sample case_0 output (first 200 chars):
[CUỐI TUẦN NÀY ĂN BÁNH CHO ĐÁNG YÊU NHA ✨]

Team mê bánh ơi, Moon Oven Bakery vừa bật deal siêu thơm trên app: Combo 6 bánh ngọt signature giảm 25%, chỉ còn 149.000đ, lại còn freeship trong 3km. Một b


## Chạy eval

In [13]:
import re
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

from metrics import JudgeLLM, run_batch

load_dotenv(find_dotenv(usecwd=True), override=True, encoding="utf-8")

judge_llm = JudgeLLM(_judge, _judge_model_id)
print(f"JudgeLLM: {judge_llm.get_model_name()}")

JudgeLLM: gpt-5.4


In [14]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

csv_path = RUNS_DIR / f"oracle_gold_{RUN_ID}.csv"

df_scores = run_batch(cases, judge_llm, progress_chunk=EVAL_PROGRESS_CHUNK)

base = pd.DataFrame(cases)
df_out = pd.concat(
    [base.reset_index(drop=True), df_scores.drop(columns=["case_id"], errors="ignore").reset_index(drop=True)],
    axis=1,
)
df_out.insert(0, "run_id",        RUN_ID)
df_out.insert(1, "judge_backend", "azure")
df_out.insert(2, "judge_model",   JUDGE_MODEL)
df_out.insert(3, "model_role",    "oracle")

df_out.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {csv_path}")

c:\Users\vqnhan\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Chunk 1–10/100 hoàn thành trong 256.9s


Chunk 11–20/100 hoàn thành trong 254.9s


Chunk 21–30/100 hoàn thành trong 233.8s


Chunk 31–40/100 hoàn thành trong 261.9s


...

Chunk 51–60/100 hoàn thành trong 252.0s


Chunk 61–70/100 hoàn thành trong 254.2s


Chunk 71–80/100 hoàn thành trong 214.2s


Chunk 81–90/100 hoàn thành trong 257.2s


Chunk 91–100/100 hoàn thành trong 259.0s
Đã lưu: D:\Github\mcs-train-content-model\results\runs\oracle_gold_20260519T065413Z.csv


In [20]:
# csv_path = RUNS_DIR / f"oracle_gold_20260519T031259Z.csv"
df_out   = pd.read_csv(csv_path, encoding="utf-8-sig")
RUN_ID   = df_out["run_id"].iloc[0]
score_cols = [c for c in df_out.columns
              if c not in {"run_id","judge_backend","judge_model","model_role",
                           "case_id","input_title","seed_content","actual_output"}]
df_scores = df_out[score_cols].copy()
print(f"Loaded df_scores from {csv_path.name}  ({len(df_out)} rows)")

summary_cols = ["faithfulness_combined", "expansion_combined", "vibe_combined"]
means   = df_scores[summary_cols].mean().to_dict()
overall = float(np.mean(list(means.values())))

summary_row = {
    "run_id":                  RUN_ID,
    "judge_backend":           "azure",
    "judge_model":             JUDGE_MODEL,
    "model_role":              "oracle",
    "local_model_id":          "gold-responses",
    "dataset_file":            str(DATASET_PATH.relative_to(ROOT)),
    "n_cases":                 len(df_scores),
    "csv_file":                str(csv_path.relative_to(ROOT)),
    "faithfulness_combined":   means["faithfulness_combined"],
    "expansion_combined":      means["expansion_combined"],
    "vibe_combined":           means["vibe_combined"],
    "overall_mean":            overall,
    "last_updated_utc":        datetime.now(timezone.utc).isoformat(),
}

new_df = pd.DataFrame([summary_row])
if SUMMARY_PATH.is_file():
    prev_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
    summary_out = pd.concat([prev_df, new_df], ignore_index=True)
else:
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    summary_out = new_df

summary_out.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
print(f"Đã cập nhật tổng hợp: {SUMMARY_PATH}")
pd.DataFrame([summary_row]).T


Loaded df_scores from oracle_gold_20260519T065413Z.csv  (100 rows)
Đã cập nhật tổng hợp: D:\Github\mcs-train-content-model\results\fnb_eval_summary.csv


,0
run_id,20260519T065413Z
judge_backend,azure
judge_model,gpt-5.4
model_role,oracle
local_model_id,gold-responses
dataset_file,dataset\fnb_dataset_test.json
n_cases,100
csv_file,results\runs\oracle_gold_20260519T065413Z.csv
faithfulness_combined,0.92391
expansion_combined,0.8341


## Kết quả chi tiết & phân phối điểm

In [21]:
print("=== Oracle score distribution ===")
df_scores[["faithfulness_combined", "expansion_combined", "vibe_combined"]].describe().round(3)

=== Oracle score distribution ===


,faithfulness_combined,expansion_combined,vibe_combined
count,100.000,100.000,100.000
mean,0.924,0.834,0.730
std,0.127,0.068,0.100
min,0.500,0.610,0.504
25%,0.865,0.820,0.661
50%,1.000,0.820,0.734
75%,1.000,0.910,0.815
max,1.000,0.910,0.968


In [22]:
# 10 case có vibe thấp nhất — giúp chẩn đoán rubric có bị mis-calibrate không
df_out[["case_id", "input_title", "vibe_combined", "vibe_llm_reason"]].sort_values("vibe_combined").head(10)

,case_id,input_title,vibe_combined,vibe_llm_reason
8,8,Viết bài Facebook content cho nhà hàng fine-di...,0.503750,"Điểm mạnh: có giá 1.990.000đ/cặp, quà tặng và ..."
55,55,Viết bài Facebook quảng bá cho thương hiệu đồ ...,0.521875,Điểm mạnh: có neo business khá rõ với combo 24...
70,70,Viết bài Facebook ngắn gọn để tuyển đối tác nh...,0.533906,Điểm mạnh: có neo business rõ với giá từ 149 t...
43,43,Viết Facebook content cho chiến dịch GDN/banne...,0.573469,"Điểm mạnh: có giá combo 199k, mốc freeship 299..."
83,83,Viết bài Facebook phản hồi review tiêu cực cho...,0.575031,"Điểm mạnh: có số rất cụ thể về đổi trả, vouche..."
87,87,Viết bài Facebook content cho thương hiệu cà p...,0.576875,"Mạnh ở chỗ có số rất cụ thể: giá 39k, KPI, ngâ..."
35,35,Viết bài Facebook content cho thương hiệu trà ...,0.580000,"Điểm mạnh: có offer và KPI rất cụ thể, neo rõ ..."
86,86,Viết content Facebook ngắn gọn để đăng tải kèm...,0.589375,"Điểm mạnh: có offer rõ 14h-17h, Mua 2 Tặng 1 v..."
63,63,Viết bài Facebook khai trương cửa hàng mới cho...,0.600656,Mạnh: có deal và ngưỡng bill rất cụ thể (mua 2...
61,61,Viết bài Facebook content cho nhà hàng fine-di...,0.608750,"Điểm mạnh: có neo kinh doanh rõ với 8 món, giả..."


In [18]:
# 10 case có vibe cao nhất — xem rubric hiểu đúng phong cách mục tiêu không
df_out[["case_id", "input_title", "vibe_combined", "vibe_llm_reason"]].sort_values("vibe_combined", ascending=False).head(10)

,case_id,input_title,vibe_combined,vibe_llm_reason
57,57,Viết bài Facebook Ads cho thương hiệu cà phê t...,0.968125,"Mạnh ở chỗ rất grounded: giá 49k, freeship 3km..."
65,65,Viết bài Facebook content cho thương hiệu trà ...,0.938125,"Mạnh: giọng founder rất thật, có giá/deal/KPI/..."
90,90,Viết bài Facebook content cho thương hiệu fast...,0.938125,"Mạnh ở giọng founder rất thật, có giá/voucher/..."
10,10,Viết bài Facebook cho thương hiệu cloud kitche...,0.915000,"Mạnh: có giá 99k, quyền lợi rõ, KPI cụ thể và ..."
44,44,Viết bài Facebook theo góc nhìn founder startu...,0.908125,"Mạnh: có số rất cụ thể như GMV 186 triệu, ROAS..."
36,36,Viết caption Facebook cho thương hiệu rượu van...,0.895625,"Mạnh: có founder tone khá thật, offer rõ với g..."
88,88,Viết bài Facebook cho thương hiệu nhà hàng buf...,0.885000,"Mạnh: giọng founder thẳng, có giá 299k, deal n..."
5,5,Viết bài Facebook content cho thương hiệu snac...,0.885000,"Mạnh: neo business rất chắc với giá 189k, mua ..."
67,67,Viết bài Facebook ra mắt sản phẩm mới cho một ...,0.884594,"Mạnh ở chỗ bám số rất chắc: giá, combo, KPI, b..."
95,95,Viết bài Facebook cho thương hiệu gia vị/nước ...,0.861875,"Mạnh: có giá bậc thang, freeship, neo AOV/KPI ..."


In [24]:
# Đọc summary tổng hợp — so sánh oracle vs baseline vs trained khi có đủ runs
if SUMMARY_PATH.is_file():
    df_all = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
    cols = ["model_role", "local_model_id", "run_id",
            "faithfulness_combined", "expansion_combined", "vibe_combined", "overall_mean"]
    display_cols = [c for c in cols if c in df_all.columns]
    print(df_all[display_cols].sort_values(["model_role", "run_id"]).to_string(index=False))

model_role local_model_id           run_id  faithfulness_combined  expansion_combined  vibe_combined  overall_mean
    oracle gold-responses 20260519T031259Z               0.922781              0.8233       0.665792      0.803958
    oracle gold-responses 20260519T065413Z               0.923910              0.8341       0.730498      0.829503
    oracle gold-responses 20260519T065413Z               0.923910              0.8341       0.730498      0.829503
